In [5]:
# 1. Mount Google Drive (Watch for the authorization popup!)
print("⏳ Mounting Google Drive... (Please allow access if prompted)")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Google Drive mounted successfully!\n")

# 2. Install YOLO (Removed '-q' to see installation logs and catch errors)
print("⏳ Installing Ultralytics YOLO...")
!pip install ultralytics
print("✅ Installation complete!\n")

# 3. Transfer Data and Weights (Using rsync to display a live progress bar)
print("⏳ Copying dataset_v5.zip from Drive... (Watch the progress bar below):")
!rsync -ah --progress "/content/drive/MyDrive/RoadEye_V5_Workspace/dataset_v5.zip" "/content/"

print("\n⏳ Copying best_v4.pt from Drive...")
!rsync -ah --progress "/content/drive/MyDrive/RoadEye_V5_Workspace/best_v4.pt" "/content/"
print("✅ All files copied successfully!\n")

# 4. Extract Dataset (Keeping it quiet but adding start/end checkpoints)
print("⏳ Unzipping dataset_v5.zip... (This usually takes 1-3 minutes)")
!unzip -o -q /content/dataset_v5.zip -d /content/
print("✅ Dataset unzipped and loaded! Environment is 100% ready for training.")

⏳ Mounting Google Drive... (Please allow access if prompted)
Mounted at /content/drive
✅ Google Drive mounted successfully!

⏳ Installing Ultralytics YOLO...
✅ Installation complete!

⏳ Copying dataset_v5.zip from Drive... (Watch the progress bar below):
sending incremental file list
dataset_v5.zip
        679.86M 100%   50.08MB/s    0:00:12 (xfr#1, to-chk=0/1)

⏳ Copying best_v4.pt from Drive...
sending incremental file list
best_v4.pt
        103.94M 100%   20.86MB/s    0:00:04 (xfr#1, to-chk=0/1)
✅ All files copied successfully!

⏳ Unzipping dataset_v5.zip... (This usually takes 1-3 minutes)
✅ Dataset unzipped and loaded! Environment is 100% ready for training.


In [4]:
import yaml

yaml_content = {
    "path": "/content/dataset_v5_merged_clean",
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {
        0: "Crack",
        1: "Pothole"
    }
}

with open("/content/roadeye_v5_colab.yaml", "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print("✅ YAML file generated for Colab!")

✅ YAML file generated for Colab!


In [1]:
import warnings
warnings.filterwarnings("ignore")
!pip install ultralytics
from ultralytics import YOLO

# ============================================================
# RoadEye V5 — Japan + India Domain Adaptation
# Final Stable Training Config (Colab Free + T4 Optimized)
# ============================================================

# 1. Load V4 best checkpoint (Japan baseline)
model = YOLO("/content/best_v4.pt")

# 2. Start the training process
results = model.train(

    # Dataset
    data="/content/roadeye_v5_colab.yaml",

    # Saving (Directly to Google Drive)
    project="/content/drive/MyDrive/RoadEye_V5_Workspace/Runs",
    name="V5_Domain_Adaptation_Final",

    # Core Training
    epochs=80,
    imgsz=1024,
    batch=4,                 # Safe for Colab T4 (15GB)

    # Optimizer
    optimizer="SGD",
    lr0=0.0025,
    lrf=0.1,
    momentum=0.937,
    weight_decay=0.0005,
    cos_lr=True,

    # Warmup (Reduced for Fine-Tuning)
    warmup_epochs=1.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.05,

    # Partial Freeze (Keep low-level road features stable)
    freeze=4,

    # Augmentations
    mosaic=0.75,
    close_mosaic=20,
    mixup=0.0,
    copy_paste=0.10,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.25,
    degrees=7,
    translate=0.08,
    scale=0.35,
    shear=2.0,
    perspective=0.0002,
    flipud=0.0,
    fliplr=0.5,

    # Loss Configuration
    cls=0.7,
    box=7.5,
    dfl=1.5,

    # System / Stability
    amp=True,
    cache=False,
    patience=30,
    deterministic=True,
    seed=42,
    workers=2,
    device=0,

    # Logging / Saving
    save=True,
    save_period=5,
    plots=True,
    val=True,
)

print("\n✅ RoadEye V5 Training Complete!")

try:
    print(f"mAP50     : {results.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"mAP50-95  : {results.results_dict['metrics/mAP50-95(B)']:.4f}")
except:
    print("Metrics will appear inside the training logs and results folder.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.5 MB/s eta 0:00:00


KeyboardInterrupt: 

In [2]:
from ultralytics import YOLO

# ============================================================
# RoadEye V5 — Resume Training Protocol for Teammates
# ============================================================

# 1. Load the LAST checkpoint from the shared Google Drive
# Make sure the exact path matches where the first run was saved
last_checkpoint_path = "/content/drive/MyDrive/RoadEye_V5_Workspace/Runs/V5_Domain_Adaptation_Final/weights/last.pt"

model = YOLO(last_checkpoint_path)

# 2. Resume training exactly from where the previous session stopped
# YOLOv8 automatically loads all hyperparameters from the saved checkpoint
results = model.train(resume=True)

print("\n✅ RoadEye V5 Resume Training Complete!")

AttributeError: partially initialized module 'torch' has no attribute 'distributed' (most likely due to a circular import)